# Module 07 — GPU Hardware Selection

This notebook compares representative NVIDIA T4, A100, and H100 constraints, estimates model memory fit, visualizes simplified roofline ceilings, and optionally probes the CUDA device attached to the notebook.

The central rule is: **capacity is a gate; bandwidth, compute, interconnect, and cost determine speed after the workload fits.**

## Learning goals

By the end, you should be able to:

- compare memory capacity, bandwidth, compute, and interconnect;
- estimate weight and KV-cache memory for a serving configuration;
- classify a workload with a simplified roofline;
- verify the actual GPU instead of trusting the requested runtime;
- explain why an end-to-end vLLM benchmark is still required.

In [ ]:
from pathlib import Path
import sys

# Make the companion Python module importable from either the repository
# root or a notebook launched inside code/module-07.
candidates = [Path.cwd() / 'code' / 'module-07', Path.cwd()]
module_dir = next(path for path in candidates if (path / 'hardware_selection.py').exists())
sys.path.insert(0, str(module_dir))

from hardware_selection import GPU_SPECS, estimate_budget, print_budget, print_specs

import matplotlib.pyplot as plt
import numpy as np

print('Companion module:', module_dir / 'hardware_selection.py')

## Experiment A — compare the theoretical hardware

These are representative published ceilings, not measured application results. The balance point is the arithmetic intensity where the simplified memory roof meets the dense FP16/BF16 Tensor Core roof.

In [ ]:
print_specs()

In [ ]:
# A compact visual comparison of the four main hardware axes.
names = [spec.name.replace(' 80GB SXM', '') for spec in GPU_SPECS]
memory = [spec.memory_gb for spec in GPU_SPECS]
bandwidth = [spec.bandwidth_gbs / 1000 for spec in GPU_SPECS]
compute = [spec.fp16_tflops for spec in GPU_SPECS]
balance = [spec.fp16_balance for spec in GPU_SPECS]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, values, title, ylabel in zip(
    axes,
    [memory, bandwidth, compute, balance],
    ['VRAM', 'Memory bandwidth', 'Dense FP16/BF16 peak', 'Balance point'],
    ['GB', 'TB/s', 'TFLOPS', 'FLOPs/byte'],
):
    bars = ax.bar(names, values, color=['#38bdf8', '#2dd4bf', '#a78bfa'])
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.grid(axis='y', alpha=0.25)
    ax.bar_label(bars, fmt='%.1f', padding=3, fontsize=9)
    ax.tick_params(axis='x', rotation=35)
fig.suptitle('Representative GPU trade-offs', y=1.03, fontsize=15, fontweight='bold')
fig.tight_layout()
plt.show()

## Experiment B — estimate model and KV-cache fit

The default example models a 7B FP16 model with 2 GB reserved for runtime/workspace and 0.125 GB of KV cache per 1,000 tokens. Treat this as a planning model: real checkpoints, quantization metadata, allocator behavior, and vLLM settings must be measured.

In [ ]:
parameters_b = 7
weight_bytes = 2.0       # FP16/BF16
reserve_gb = 2.0
kv_gb_per_1k_tokens = 0.125
tokens = 4_096

print_budget(
    parameters_b,
    weight_bytes,
    reserve_gb,
    kv_gb_per_1k_tokens,
    tokens,
)

In [ ]:
# Context length changes the KV-cache term while weights stay fixed.
context_lengths = np.array([512, 2_048, 4_096, 8_192, 16_384])
fig, ax = plt.subplots(figsize=(9, 5))
for spec in GPU_SPECS:
    usable = np.full_like(context_lengths, spec.usable_gb, dtype=float)
    totals = np.array([
        estimate_budget(
            parameters_b, weight_bytes, reserve_gb,
            kv_gb_per_1k_tokens, int(context),
        )['total_gb']
        for context in context_lengths
    ])
    ax.plot(context_lengths, totals, marker='o', label=f'{spec.name} estimate')
    ax.plot(context_lengths, usable, linestyle='--', alpha=0.55, label=f'{spec.name} usable 90%')

ax.set_xscale('log', base=2)
ax.set_xlabel('Total active tokens')
ax.set_ylabel('GB')
ax.set_title('Estimated memory demand versus usable VRAM')
ax.grid(alpha=0.25)
ax.legend(ncol=2, fontsize=8)
plt.show()

### Change the quantization assumption

Run the next cell with `weight_bytes = 0.5` to approximate INT4 weight storage. The estimate will show why quantization can turn capacity from a hard blocker into a feasible configuration, while leaving KV-cache and runtime costs in place.

In [ ]:
for dtype_name, bytes_per_weight in [('FP16/BF16', 2.0), ('INT8', 1.0), ('INT4', 0.5)]:
    estimate = estimate_budget(
        parameters_b, bytes_per_weight, reserve_gb,
        kv_gb_per_1k_tokens, tokens,
    )
    fits = [spec.name for spec in GPU_SPECS if estimate['total_gb'] <= spec.usable_gb]
    print(f'{dtype_name:10s}: {estimate["total_gb"]:6.2f} GB; fits: {fits}')

## Experiment C — simplified roofline comparison

For arithmetic intensity `AI`, the idealized ceiling is `min(compute_peak, AI * bandwidth)`. This chart helps explain why a GPU can have much higher peak compute without improving a low-intensity decode workload by the same factor.

In [ ]:
arithmetic_intensity = np.logspace(-2, 3, 300)
fig, ax = plt.subplots(figsize=(9, 5))
for spec, color in zip(GPU_SPECS, ['#38bdf8', '#2dd4bf', '#a78bfa']):
    memory_roof = arithmetic_intensity * spec.bandwidth_gbs / 1000
    ceiling = np.minimum(spec.fp16_tflops, memory_roof)
    ax.loglog(arithmetic_intensity, ceiling, color=color, label=spec.name)
    ax.axvline(spec.fp16_balance, color=color, alpha=0.25, linestyle='--')

ax.set_xlabel('Arithmetic intensity (FLOPs/byte)')
ax.set_ylabel('Idealized ceiling (TFLOPS)')
ax.set_title('Dense FP16/BF16 roofline comparison')
ax.grid(which='both', alpha=0.2)
ax.legend()
plt.show()

## Experiment D — inspect the actual CUDA device

This cell is safe to run on CPU-only environments: it reports that the probe was skipped. On Lightning, it records the attached GPU, compute capability, CUDA runtime, PyTorch version, total memory, free memory, and SM count.

In [ ]:
import torch

if torch.cuda.is_available():
    from hardware_selection import probe_device
    probe_device(measure_h2d=False, transfer_mb=256, repeats=5)
else:
    print('CUDA is unavailable; run this cell on a Lightning GPU runtime.')

### Optional Experiment E — host-to-device transfer

This measures a pinned host-to-device copy. It is a PCIe/system-path measurement, not HBM bandwidth, so do not compare it directly with the T4's 320 GB/s memory specification.

In [ ]:
if torch.cuda.is_available():
    from hardware_selection import probe_device
    probe_device(measure_h2d=True, transfer_mb=256, repeats=20)
else:
    print('Skipped: CUDA is unavailable.')

## Optional T4 roofline microbenchmarks

The companion `roofline_benchmarks.py` script measures a large FP32 vector add, an FP16 GEMM, and a fixed-weight projection across token dimensions. Run it from a T4 runtime when you want device timings:

```bash
python code/module-07/roofline_benchmarks.py --plot projection_tflops.png
```

Use CUDA events, warmups, and medians. The results are hypotheses about the bottleneck; profiler counters are needed to confirm actual DRAM and Tensor Core utilization.

In [ ]:
# Notebook-friendly version of the optional projection experiment.
# Keep False for a quick documentation run; set True on a T4 runtime.
RUN_CUDA_BENCHMARKS = False

if RUN_CUDA_BENCHMARKS:
    from roofline_benchmarks import projection_experiment
    results = projection_experiment(
        torch.device('cuda:0'),
        batches=[1, 4, 16, 32, 64, 128, 256],
        hidden_size=4096,
        warmup=10,
        repeats=30,
    )
    batches = [int(result.label.rsplit(' ', 1)[-1]) for result in results]
    tflops = [result.tflops for result in results]
    plt.figure(figsize=(8, 4))
    plt.plot(batches, tflops, marker='o')
    plt.xlabel('Input rows / token positions')
    plt.ylabel('Achieved TFLOPS')
    plt.title('T4 projection efficiency versus batch size')
    plt.grid(alpha=0.3)
    plt.show()
else:
    print('CUDA roofline benchmarks disabled. Set RUN_CUDA_BENCHMARKS = True on a T4.')

## Interpretation checklist

Write down:

1. the smallest GPU that fits the model with runtime headroom;
2. whether your workload is likely bandwidth- or compute-sensitive;
3. whether the selected dtype has an efficient kernel on the GPU;
4. whether PCIe/NVLink matters for the parallelism plan;
5. TTFT, inter-token latency, throughput, memory, and cost measurements from an end-to-end vLLM run.

A peak specification is a ceiling. The serving configuration and request mix determine how close the workload gets to it.